In [1]:
import pandas as pd
import numpy as np
import re
import hashlib
from pathlib import Path

### The reason you need chunks:

LLMs have context limits
You cannot efficiently pass every HP Academy transcript into the model every time.
Search becomes more accurate
If the user asks:
“What fuel system upgrades do I need for 600 whp on a B58?”
RedlineIQ can retrieve only the transcript chunks about fueling, injectors, ethanol, pumps, tuning, etc.
Answers become grounded
The model can answer based on actual course content instead of guessing.
You can attach metadata
Each chunk keeps its course name, module title, lesson title, URL, source type, and chunk index. That means RedlineIQ can later say:
“This recommendation is based on HP Academy course transcript X, lesson Y.”
Vector databases work on chunks
You would embed each chunk_text, store it in PostgreSQL + pgvector, Pinecone, Chroma, Weaviate, etc., and retrieve the most relevant chunks for a user question.

In [2]:
def stable_id(*parts: Any, prefix: str = '') -> str:
    raw = '||'.join('' if p is None else str(p) for p in parts)
    h = hashlib.md5(raw.encode('utf-8')).hexdigest()[:12]
    return f'{prefix}{h}' if prefix else h

def normalize_system_category(category: Any, text: str) -> str:
    c = category if pd.notna(category) else ''
    c_l = c.lower()
    if c_l in CATEGORY_MAP:
        return CATEGORY_MAP[c_l]
    t = text.lower()
    for label, kws in PART_KEYWORDS.items():
        if any(k in t for k in kws):
            if label in ['turbo', 'intake']:
                return 'engine_power'
            return label
    return 'other'
    
def safe_literal_list(x: Any) -> list:
    if pd.isna(x) or x in ('', '[]'):
        return []
    try:
        val = ast.literal_eval(str(x))
        return val if isinstance(val, list) else [val]
    except Exception:
        return []


def first_image(x: Any) -> str | None:
    urls = safe_literal_list(x)
    for url in urls:
        u = str(url)
        if any(ext in u.lower() for ext in ['.jpg', '.jpeg', '.png', '.webp']):
            return u
    return None

def extract_subsystem(text: str) -> str:
    t = text.lower()
    for label, kws in PART_KEYWORDS.items():
        if any(k in t for k in kws):
            return label
    return 'other'

def extract_fitment_flags(text: str) -> dict[str, Any]:
    t = text.lower()
    years = [int(y) for y in re.findall(r'\b(20[12][0-9]|2030)\b', t)]
    
    # Filter years to valid A90 Supra range (2019-2026)
    valid_years = [y for y in years if 2019 <= y <= 2026]
    
    # Use A90 Supra default range if no valid years found
    if valid_years:
        model_year_min = min(valid_years)
        model_year_max = max(valid_years)
    else:
        model_year_min = 2019
        model_year_max = 2026

    car_fitment_flags = {
        'fits_gr_supra': bool(re.search(r'gr supra|mkv supra|mk5 supra|j29|a90|a91', t)),
        'fits_a90': 'a90' in t or 'j29' in t,
        'fits_a91': 'a91' in t,
        'fits_b58': 'b58' in t or '3.0' in t,
        'fits_b48': 'b48' in t or '2.0' in t,
        'fits_3_0': 'b58' in t or '3.0' in t,
        'fits_2_0': 'b48' in t or '2.0' in t,
        'model_year_min': model_year_min,
        'model_year_max': model_year_max,
    }
    
    return car_fitment_flags

def extract_numeric_claims(text: str) -> dict[str, Any]:
    t = text.lower()
    hp_vals = [int(v.replace('+', '')) for v, _ in re.findall(r'(\+?\d{1,4})\s?(whp|hp|horsepower)\b', t)]
    tq_vals = [int(v.replace('+', '')) for v, _ in re.findall(r'(\+?\d{1,4})\s?(tq|torque|lb[- ]?ft|ft[- ]?lb)\b', t)]
    weight_matches = re.findall(r'([+-]?\d+(?:\.\d+)?)\s?(lbs?|pounds|kg)\b', t)
    weights_lbs = []
    for val, unit in weight_matches:
        f = float(val)
        if unit == 'kg':
            f *= 2.20462
        weights_lbs.append(f)


    extracted_numeric_claims = {
        'claimed_hp_gain_min': min(hp_vals) if hp_vals else None,
        'claimed_hp_gain_max': max(hp_vals) if hp_vals else None,
        'claimed_tq_gain_min': min(tq_vals) if tq_vals else None,
        'claimed_tq_gain_max': max(tq_vals) if tq_vals else None,
        'weight_delta_lbs': weights_lbs[0] if weights_lbs else None,
        'hp_gain_source': 'stated' if hp_vals else 'missing',
    }

    return 


In [3]:
def clean_text(x: Any):
    if pd.isna(x):
        return None
    string = str(x)
    #removing html tags and non-breaking spaces, normalizing whitespace
    string = re.sub(r'<[^>]+>', ' ', string)
    string = string.replace('\xa0', ' ')
    string = re.sub(r'\s+', ' ', string).strip()

    return string or None


def parse_lap_time_seconds(text: str) -> float | None:
    # catches formats like 57.109, 1:48.30, 2:01.123; avoids dates by context-limiting in lap posts only
    candidates = re.findall(r'(?<!\d)(?:(\d{1,2}):)?(\d{1,2})\.(\d{2,3})(?!\d)', text)
    if not candidates:
        return None
    vals = []
    for mins, sec, frac in candidates:
        s = int(sec) + int(frac) / (1000 if len(frac) == 3 else 100)
        if mins:
            s += int(mins) * 60
        if 30 <= s <= 300:
            vals.append(s)
    return min(vals) if vals else None


def extract_track(text: str) -> str | None:
    t = text.lower()
    for tr in KNOWN_TRACKS:
        if tr in t:
            return tr.title()
    return None


def extract_part_mentions(text: str) -> list[str]:
    t = text.lower()
    mentions = []
    for label, kws in PART_KEYWORDS.items():
        if any(k in t for k in kws):
            mentions.append(label)
    return sorted(set(mentions))

In [4]:
RAW_DIR = Path('../backend/app/data')
OUT_DIR = RAW_DIR / 'processed/redlineiq_cleaned'
OUT_DIR.mkdir(exist_ok=True)

PATHS = {
    'hpacademy': RAW_DIR / 'ingestion/raw/hp_academy_data_raw/hpacademy_course_content.csv',
}


In [7]:
def clean_hpacademy() -> pd.DataFrame:
    hpa = pd.read_csv(PATHS['hpacademy'])
    for col in ['course_type', 'section', 'course_name', 'module_title', 'item_type', 'item_title', 'lesson_title', 'transcript']:
        hpa[col] = hpa[col].apply(clean_text)
    hpa = hpa[hpa['transcript'].notna()].copy()
    hpa['doc_id'] = hpa.apply(lambda r: stable_id(r['course_name'], r['item_id'], r['item_title'], prefix='hpa_'), axis=1)
    hpa['source_type'] = 'hpacademy_course_transcript'
    hpa['duration_minutes'] = pd.to_numeric(hpa['duration_or_score'], errors='coerce')
    rows = []
    for _, r in hpa.iterrows():
        text = r['transcript'] or ''
        #Create chunks of 1200 characters with 200 character overlap between chunks
        chunks = [text[i:i+1200] for i in range(0, len(text), 1000)] or ['']
        #Create one row per chunk for a structured knowledge base
        for idx, ch in enumerate(chunks):
            rows.append({
                'chunk_id': stable_id(r['doc_id'], idx, prefix='hpa_chunk_'),
                'doc_id': r['doc_id'],
                'source_type': r['source_type'],
                'course_type': r['course_type'],
                'course_name': r['course_name'],
                'module_title': r['module_title'],
                'lesson_title': r['lesson_title'] or r['item_title'],
                'item_url': r['item_url'],
                'video_url': r['video_url'],
                'chunk_index': idx,
                'chunk_text': ch,
            })
   
    expected_cols = [
            "chunk_id",
            "doc_id",
            "source_type",
            "course_type",
            "course_name",
            "module_title",
            "lesson_title",
            "item_url",
            "video_url",
            "chunk_index",
            "chunk_text",]
   
    chunks = pd.DataFrame(rows)
    chunks = chunks[expected_cols].drop_duplicates(
        subset=["chunk_id"],
        keep="first"
    )

    chunks.to_csv(OUT_DIR / 'hpacademy_rag_chunks.csv', index=False)
    return chunks

In [8]:
data = clean_hpacademy()

In [9]:
data

,chunk_id,doc_id,source_type,course_type,course_name,module_title,lesson_title,item_url,video_url,chunk_index,chunk_text
0,hpa_chunk_937a693fc9b1,hpa_9ac79728bc58,hpacademy_course_transcript,EFI Tuning,Bare Minimum Tuning Knowledge,NaN,Understanding Fuel Tuning,https://www.hpacademy.com/dashboard/courses/ba...,https://player.vimeo.com/video/139423872,0,- I'm Andre from the High Performance Academy ...
1,hpa_chunk_1c31065bda6a,hpa_9ac79728bc58,hpacademy_course_transcript,EFI Tuning,Bare Minimum Tuning Knowledge,NaN,Understanding Fuel Tuning,https://www.hpacademy.com/dashboard/courses/ba...,https://player.vimeo.com/video/139423872,1,some insight into what it is professional eng...
2,hpa_chunk_bcc060a87a68,hpa_9ac79728bc58,hpacademy_course_transcript,EFI Tuning,Bare Minimum Tuning Knowledge,NaN,Understanding Fuel Tuning,https://www.hpacademy.com/dashboard/courses/ba...,https://player.vimeo.com/video/139423872,2,sic fundamentals behind EFI tuning. This is a ...
3,hpa_chunk_039ea73813ac,hpa_9ac79728bc58,hpacademy_course_transcript,EFI Tuning,Bare Minimum Tuning Knowledge,NaN,Understanding Fuel Tuning,https://www.hpacademy.com/dashboard/courses/ba...,https://player.vimeo.com/video/139423872,3,"stomers don't trust tuners, they don't underst..."
4,hpa_chunk_5d027dc6a6f3,hpa_9ac79728bc58,hpacademy_course_transcript,EFI Tuning,Bare Minimum Tuning Knowledge,NaN,Understanding Fuel Tuning,https://www.hpacademy.com/dashboard/courses/ba...,https://player.vimeo.com/video/139423872,4,w much fuel we're adding to the engine. And th...
...,...,...,...,...,...,...,...,...,...,...,...
17777,hpa_chunk_30e36fe1337c,hpa_e77e9c6758c4,hpacademy_course_transcript,Automotive CAD,Practical 3D Scanning,4 Step Process,Setup in CAD,https://www.hpacademy.com/dashboard/courses/pr...,https://player.vimeo.com/video/1027056986,4,"a single scan, we should first open the step ..."
17778,hpa_chunk_1c3ffe1a080d,hpa_d9320c245097,hpacademy_course_transcript,Automotive CAD,Practical 3D Scanning,Conclusion,Conclusion,https://www.hpacademy.com/dashboard/courses/pr...,https://player.vimeo.com/video/1027057093,0,This brings us to the end of our 3D scanning c...
17779,hpa_chunk_c8662c489ecb,hpa_d9320c245097,hpacademy_course_transcript,Automotive CAD,Practical 3D Scanning,Conclusion,Conclusion,https://www.hpacademy.com/dashboard/courses/pr...,https://player.vimeo.com/video/1027057093,1,"s of 3D scanning in particular, we need to con..."
17780,hpa_chunk_abed575ea324,hpa_d9320c245097,hpacademy_course_transcript,Automotive CAD,Practical 3D Scanning,Conclusion,Conclusion,https://www.hpacademy.com/dashboard/courses/pr...,https://player.vimeo.com/video/1027057093,2,using them in CAD. Thanks again for taking th...


In [ ]:
# Check for chunks duplications and remove if necessary



hpa = pd.read_csv("../data/cleaned/hpacademy_rag_chunks.csv")

print(hpa.shape)
print("unique chunk_ids:", hpa["chunk_id"].nunique())
print("duplicate chunk_ids:", hpa["chunk_id"].duplicated().sum())

print(
    hpa[hpa["chunk_id"].duplicated(keep=False)]
    .sort_values("chunk_id")
    [["chunk_id", "course_name", "module_title", "lesson_title", "chunk_index"]]
    .head(20)
)

(17782, 11)
unique chunk_ids: 16881
duplicate chunk_ids: 901
                     chunk_id                     course_name  \
6243   hpa_chunk_00001579837d      Data Analysis Fundamentals   
14264  hpa_chunk_00001579837d      Data Analysis Fundamentals   
6394   hpa_chunk_005589697ee3      Data Analysis Fundamentals   
14415  hpa_chunk_005589697ee3      Data Analysis Fundamentals   
4700   hpa_chunk_00ac6652b92e  CAN Bus Communications Decoded   
9485   hpa_chunk_00ac6652b92e  CAN Bus Communications Decoded   
6593   hpa_chunk_00c5a96c9082      Data Analysis Fundamentals   
14614  hpa_chunk_00c5a96c9082      Data Analysis Fundamentals   
6407   hpa_chunk_0120b8de2dec      Data Analysis Fundamentals   
14428  hpa_chunk_0120b8de2dec      Data Analysis Fundamentals   
6665   hpa_chunk_013ff9d891e8      Data Analysis Fundamentals   
14686  hpa_chunk_013ff9d891e8      Data Analysis Fundamentals   
6383   hpa_chunk_01406c3b96d4      Data Analysis Fundamentals   
14404  hpa_chunk_01406c3b96d4